# Fraud Detection in Loan Applications: Data Mining Analysis

## Index (Table of Contents)

- [Abstract](#abstract)
- [1. Introduction](#1-introduction)
  - [Background](#background)
  - [Objectives](#objectives)
  - [Dataset Overview](#dataset-overview)
- [Setup](#setup)
- [2. Methodology](#2-methodology)
  - [2.1 Data Preprocessing](#21-data-preprocessing)
  - [2.2 Missing Value Imputation](#22-missing-value-imputation)
  - [2.3 Feature Engineering](#23-feature-engineering)
  - [2.4 Class Imbalance Handling](#24-class-imbalance-handling)
  - [2.5 Feature Selection](#25-feature-selection)
- [3. Modeling](#3-modeling)
  - [3.1 Train/Test Split](#31-train-test-split)
  - [3.2 Models & Hyperparameter Tuning](#32-models--hyperparameter-tuning)
  - [3.2.1 Model variants and where they were applied](#321-model-variants-and-where-they-were-applied)
  - [3.3 Logistic Regression](#33-logistic-regression)
    - [3.3.1 Logistic Regression](#331-logistic-regression)
    - [3.3.2 Logistic Regression w/ Stepwise](#332-logistic-regression-w-stepwise)
    - [3.3.3 Logistic Regression w/ Stepwise and SMOTE Resampling](#333-logistic-regression-w-stepwise-and-smote-resampling)
  - [3.4 K-Nearest Neighbors (KNN)](#34-k-nearest-neighbors-knn)
    - [3.4.1 KNN](#341-knn)
    - [3.4.2 KNN w/ Stepwise](#342-knn-w-stepwise)
    - [3.4.3 KNN w/ Stepwise and SMOTE](#343-knn-w-stepwise-and-smote)
  - [3.5 Decision Tree](#35-decision-tree)
    - [3.5.1 Decision Tree](#351-decision-tree)
    - [3.5.2 Decision Tree w/ Grid Search](#352-decision-tree-w-grid-search)
    - [3.5.3 Decision Tree w/ Grid Search and SMOTE](#353-decision-tree-w-grid-search-and-smote)
  - [3.6 Random Forest](#36-random-forest)
    - [3.6.1 Random Forest w/ GridSearch](#361-random-forest-w-gridsearch)
    - [3.6.2 Random Forest w/ GridSearch and SMOTE](#362-random-forest-w-gridsearch-and-smote)
  - [3.7 Effects of SMOTE and Feature Selection](#37-effects-of-smote-and-feature-selection)
- [4. Evaluation Strategy](#4-evaluation-strategy)
- [5. Results](#5-results)
  - [5.1 Model Performance (selected metrics for fraud class)](#51-model-performance-selected-metrics-for-fraud-class)
  - [5.2 Error Analysis](#52-error-analysis)
- [6. Discussion](#6-discussion)
- [7. Recommended Evaluation Plots](#7-recommended-evaluation-plots)
- [8. Conclusions and Future Work](#8-conclusions-and-future-work)
- [Appendices & Data Treatment](#appendices--data-treatment)

---

## Abstract

This report presents a comprehensive analysis of fraud detection in loan applications using multiple machine learning techniques. We evaluated Logistic Regression, K-Nearest Neighbors (KNN), Decision Trees, Random Forests, and Gradient Boosting, together with preprocessing steps such as KNN-based imputation, one-hot encoding, standardization, sequential feature selection and SMOTE oversampling. The objective was to maximize detection performance for the fraud class (class 1), prioritizing F1 for fraud.

---

## 1. Introduction

### Background
Fraudulent loan applications cause monetary loss and operational inefficiency for financial institutions. Automated detection using data mining techniques can scale investigative capacity while maintaining detection quality.

### Objectives
- Develop and evaluate multiple machine learning models for fraud detection.
- Compare algorithms and preprocessing choices.
- Assess the effect of feature selection and class balancing on metrics relevant to fraud detection.
- Provide recommendations for deployment and future work.

### Dataset Overview
- Observations: ~42,691 loan applications (after cleaning)  
- Target: `fraud_flag` (0 = legitimate, 1 = fraudulent)  
- Feature groups: identifiers/metadata, loan characteristics, financial profile, demographics.

---

### Setup
- Importing all requisite libraries.

In [2]:
# basic libraries
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning
# suppress all warnings for cleaner notebook output
warnings.filterwarnings('ignore')
# specifically silence scikit-learn convergence warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# data preparation libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold

# prediction models libraries
from sklearn.metrics import classification_report
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn import linear_model
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier


---

## 2. Methodology

### 2.1 Data Preprocessing
- **Duplicate removal** via `drop_duplicates()`.
- **Categorical standardization**: Capitalized and stripped text values for categorical columns.
- **Feature elimination**: Removed features with missingness/redundant, leakage risk, redundancy or artifical data (e.g., `monthly_income`, `cibil_score`, `loan_amount_requested`, `application_date`, `data_batch_id`, `loan_type_*`).
- **Outlier removal**: Applied IQR method on `loan_tenure_months` (remove > Q3 + 1.5*IQR).



In [3]:
df_train = pd.read_csv('train.csv')

# keep only distinct rows, removing duplicated ones
df_train = df_train.drop_duplicates()

# removing unwanted object features
categ_cols = ['purpose_of_loan', 'employment_status', 'property_ownership_status']
df_train[categ_cols] = df_train[categ_cols].astype('category')

# capitalizing category features
for column in categ_cols:
    df_train[column] = df_train[column].str.capitalize()
    df_train[column] = df_train[column].str.strip()

# removing untrustable dummy columns
dummyDrop = [col for col in df_train.columns if 'loan_type' in col]
df_train = df_train.drop(columns=dummyDrop)

In [4]:
df_train = df_train.drop(
    columns=['monthly_income', # missing values and redundant with 'yearly_income'
             'cibil_score', # perfectly normal, likely not real
             'loan_amount_requested', # redundant with 'loan_amount_usd'
             'application_date', # decided to not be used
             'application_id', # irrelevant for prediction
             'customer_id', # irrelevant for prediction
             'residential_address', # decided to not be used
             'Unnamed: 0', # index
             'data_batch_id' # irrelevant
             ])


# removing untrustable dummy columns
dummyDrop = [col for col in df_train.columns if 'loan_type' in col]
df_train = df_train.drop(columns=dummyDrop)

In [5]:
# removing outliers by the loan tenure months
q1 = df_train['loan_tenure_months'].quantile(0.25)
q3 = df_train['loan_tenure_months'].quantile(0.75)
iqr = q3-q1
upper_bound = q3 + iqr*1.5

# removing outliers
df_train = df_train[df_train['loan_tenure_months']<=upper_bound]

### 2.2 Missing Value Imputation
- **Gender**: KNN classifier (k=3) trained on `debt_to_income_ratio`, `applicant_age`, `yearly_income`, `annual_bonus`. Predicted binary `gender_*` for missing cases.
- **Number of dependents**: KNN (k=3) with the same features for imputation.

Rationale: KNN leverages local similarity in financial/demographic space to infer missing categorical/numeric values.

#### 2.2.1 Gender Imputation

In [6]:
## gender imputation through KNN
def gender_code(row):
    if row['gender_Male'] == 1:
        return 1
    elif row['gender_Other'] == 1:
        return 0
    else:
        None #columns as neither male or other will be treated as unkown

df_train['gender_code'] = df_train.apply(gender_code, axis=1)

# defining known and unknown gender
df_known = df_train[df_train['gender_code'].notna()]
df_unknown = df_train[df_train['gender_code'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['gender_code']

# standardizing the known and unknown selected attributes
scaler_input = StandardScaler()
x_train_scaled = scaler_input.fit_transform(x_train)
x_test_scaled = scaler_input.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_train.loc[df_train['gender_code'].isna(), 'gender_code'] = knn.predict(x_test_scaled)

# recreating the gender label, where "1" is man and "0" is woman
gender_drop = ['gender_Male', 'gender_Other']
df_train = df_train.drop(columns=gender_drop)
df_train = df_train.rename(columns={'gender_code': 'gender_male'})

# 0: woman ; 1: man
df_train['gender_male'].value_counts()

gender_male
0.0    21683
1.0    21008
Name: count, dtype: int64

#### 2.2.2 Number of Dependents

In [7]:
## number of dependents imputation through KNN

# defining known and unknown gender
df_known = df_train[df_train['number_of_dependents'].notna()]
df_unknown = df_train[df_train['number_of_dependents'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['number_of_dependents']

# standardizing the known and unknown selected attributes
scaler_input = StandardScaler()
x_train_scaled = scaler_input.fit_transform(x_train)
x_test_scaled = scaler_input.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_train.loc[df_train['number_of_dependents'].isna(), 'number_of_dependents'] = knn.predict(x_test_scaled)

### 2.3 Feature Engineering
- **One-hot encoding** using `pd.get_dummies(..., drop_first=True)` to avoid multicollinearity.
- **Standardization** for numeric features using `StandardScaler` (important for KNN and regularized logistic regression).

#### 2.3.1 One-hot Enconding

In [8]:
# transforming categorical features into dummy ones, removing one of the categories to avoid colinearility
df_train = pd.get_dummies(df_train, columns=categ_cols, drop_first=True, dtype=int)

#### 2.3.2 Train/Test Split & Scaling

In [9]:
# setting target variable
y = df_train['fraud_flag']

# setting feature columns
x = df_train.drop(columns='fraud_flag')

# setting dummy columns
dummy_cols = [col for col in x if any(sub in col for sub in categ_cols) or col == 'gender_male']
not_dummy_cols = [col for col in x if col not in dummy_cols] 

# spliting train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# scaling feature columns but dummy ones
scaler_train = StandardScaler()
x_train_scaled = pd.DataFrame(
    scaler_train.fit_transform(x_train[not_dummy_cols]), # train the model and scales at same time
    columns=not_dummy_cols, 
    index=x_train.index)

x_test_scaled = pd.DataFrame(
    scaler_train.transform(x_test[not_dummy_cols]), # train the model and scales at same time
    columns=not_dummy_cols, 
    index=x_test.index)

# concatenating the dummy cols and the scaled numeric features
x_train = pd.concat([x_train_scaled, x_train[dummy_cols]], axis=1)
x_test = pd.concat([x_test_scaled, x_test[dummy_cols]], axis=1)


### 2.4 Class Imbalance Handling
- **SMOTE** was applied only to training data to synthesize minority-class samples. This balances classes during training while preserving test-set distribution.

In [10]:
# smoted scaled 
smote = SMOTE(sampling_strategy='minority')
x_train_SMOTE, y_smote = smote.fit_resample(x_train, y_train) #smotes train only

### 2.5 Feature Selection
- **Sequential Feature Selection (SFS)** with f1 scoring (forward selection) used to create reduced feature sets for models. Used both on original and SMOTE-resampled training data.

---

## 3. Modeling

### 3.1 Train/Test Split
- Stratified split: 80% train / 20% test, `random_state=42`.
### 3.2 Models & Hyperparameter Tuning
- **Logistic Regression**: GridSearchCV across penalty (l1, l2, elasticnet, none), C (logspace), solvers (liblinear, lbfgs, saga, newton-cg), max_iter (1000–5000). `class_weight='balanced'` used.
- **KNN**: `n_neighbors` searched in range(2-30) neighbours, `weights='distance'`, 5-fold CV.
- **Decision Tree**: Grid search on `max_depth`, `min_samples_split`, `min_samples_leaf`.
- **Random Forest**: Grid search on `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `bootstrap`.

Cross-validation and GridSearchCV (scoring='f1') were used to avoid overfitting and find robust hyperparameters.

### 3.2.1 Model variants and where they were applied

To make the experimental design explicit, the table below summarizes which models were subject to systematic hyperparameter search (GridSearchCV / CV parameter sweep), which models were trained with SMOTE resampling in any experiment, and which models were evaluated with Sequential Feature Selection (SFS, stepwise).

| Model | GridSearch / CV search | SMOTE variants | Stepwise (SFS) | Notes |
|---|:---:|:---:|:---:|---|
| Logistic Regression | Yes (GridSearchCV) | Yes (both no-SMOTE and SMOTE variants reported) | Yes (LR + Stepwise variants present) | Threshold optimization can be applied to probabilistic outputs |
| K-Nearest Neighbors (KNN) | Yes (CV search over `n_neighbors`) | Yes (some runs with SMOTE; SMOTE affected neighborhood geometry) | Yes (stepwise variants reported) | Standardization required; best `k` often large (e.g., 29) |
| Decision Tree | Yes (Grid search on depth/splits) | Yes (DT + GridSearch + SMOTE variant included) | No (not typically stepwise) | Prone to overfitting without constraints |
| Random Forest | Yes (GridSearchCV) | Yes (RF + GridSearch + SMOTE variant included) | No | Feature importances used for interpretability |

This explicit mapping clarifies why some table rows (e.g., "LR + Stepwise + SMOTE" or "RF + GridSearch + SMOTE") appear: they represent separate experiments in which resampling and/or feature selection were applied prior to model training and evaluation. The main results table reports the most representative runs per model variant.


#### 3.3 Logistic Regression

Logistic Regression is a generalized linear model for binary classification that models the log-odds of the positive class as a linear function of the features. Regularization (L1, L2 and ElasticNet) was applied to control variance and handle multicollinearity; solvers were chosen according to penalty compatibility. Logistic Regression outputs calibrated probabilities; threshold optimization on the precision–recall curve is a recommended procedure for operational tuning, but it was not systematically applied across the main experiments in this report. Strengths: interpretability, fast training, and well-understood regularization behavior. Limitations: linear decision boundary may underfit complex, non-linear fraud patterns, and performance is sensitive to feature scaling and informative interaction terms.

**Hyperparameter meanings**
- **Logistic Regression**: `penalty` (type of regularization; `l1` = Lasso, `l2` = Ridge, `elasticnet` = mix), `C` (inverse of regularization strength: smaller => stronger regularization), `solver` (optimization algorithm; choose compatible solvers for each penalty), `max_iter` (max iterations to converge), `class_weight='balanced'` (adjusts class weights for imbalance).

#### 3.3.1 Logistic Regression 

In [11]:
# creating model
log_reg_model = linear_model.LogisticRegression(random_state=42, class_weight='balanced')

param_grid = [
    {
        'penalty':['l2'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['lbfgs','newton-cg'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['liblinear'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1', 'l2', 'elasticnet'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['saga'], 
        'max_iter': [1000, 3000] # Aumentar max_iter para garantir convergência do SAGA
    },
    {
        'penalty': ['none'], 
        'C': [1.0], # C é ignorado, mas deve estar presente para compatibilidade
        'solver': ['lbfgs'], 
        'max_iter': [1000, 3000]
    }]

# executing GridSearch
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_log_reg_model = grid_search.best_estimator_ #already returns the best model trained

# prediction
y_pred = best_log_reg_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred), '\n')
print("Classification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results = dict()

results['Logistic Regression'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 86 candidates, totalling 430 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           3409  3462
1            843   825 

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.50      0.61      6871
           1       0.19      0.49      0.28      1668

    accuracy                           0.50      8539
   macro avg       0.50      0.50      0.45      8539
weighted avg       0.68      0.50      0.55      8539



In [12]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.1 
 solver saga 
 max_iter 1000


#### 3.3.2 Logistic Regression w/ Stepwise

In [13]:
# setting stepwise
sfs_log_reg = SequentialFeatureSelector(
    best_log_reg_model,
    scoring='f1',
    cv=5
)

# applying the stepwise model
selected_features = sfs_log_reg.fit(x_train, y_train)

# selecting the stepwised features
x_train_stepwised = x_train[selected_features.get_feature_names_out()]
x_test_stepwised = x_test[selected_features.get_feature_names_out()]

# parameters grid
param_grid = [
    {
        'penalty':['l2'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['lbfgs','newton-cg'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['liblinear'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1', 'l2', 'elasticnet'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['saga'], 
        'max_iter': [1000, 3000] 
    },
    {
        'penalty': ['none'], 
        'C': [1.0], 
        'solver': ['lbfgs'], 
        'max_iter': [1000, 3000]
    }]

# setting GridSearch model
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# applying the GridSearch model
grid_search.fit(x_train_stepwised, y_train)
best_log_reg_model = grid_search.best_estimator_

# prediction
y_pred = best_log_reg_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Logistic Regression w/ Stepwise'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 86 candidates, totalling 430 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           2147  4724
1            528  1140

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.31      0.45      6871
           1       0.19      0.68      0.30      1668

    accuracy                           0.38      8539
   macro avg       0.50      0.50      0.38      8539
weighted avg       0.68      0.38      0.42      8539



In [14]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.1 
 solver saga 
 max_iter 1000


#### 3.3.3 Logistic Regression w/ Stepwise and SMOTE Resampling

In [15]:
# stepwising after SMOTE
selected_features_SMOTE = sfs_log_reg.fit(x_train_SMOTE, y_smote) #select features after SMOTE
x_stepwised_SMOTE = x_train_SMOTE[selected_features_SMOTE.get_feature_names_out()]

# selecting the stepwise columns for the test sample
x_test_stepwised = x_test[selected_features_SMOTE.get_feature_names_out()]

# parameters Grid
param_grid = [
    {
        'penalty':['l2'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['lbfgs','newton-cg'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['liblinear'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1', 'l2', 'elasticnet'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['saga'], 
        'max_iter': [1000, 3000] # Aumentar max_iter para garantir convergência do SAGA
    },
    {
        'penalty': ['none'], 
        'C': [1.0], # C é ignorado, mas deve estar presente para compatibilidade
        'solver': ['lbfgs'], 
        'max_iter': [1000, 3000]
    }]

# executing GridSearch
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# applying the grid search
grid_search.fit(x_stepwised_SMOTE, y_smote)
best_log_reg_model = grid_search.best_estimator_

# the smoted data should be used only for training, not for tests
y_pred = best_log_reg_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Logistic Regression w/ Stepwise and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 86 candidates, totalling 430 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           1916  4955
1            475  1193

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.28      0.41      6871
           1       0.19      0.72      0.31      1668

    accuracy                           0.36      8539
   macro avg       0.50      0.50      0.36      8539
weighted avg       0.68      0.36      0.39      8539



In [16]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.01 
 solver saga 
 max_iter 1000


#### 3.4 K-Nearest Neighbors (KNN)

KNN is a non-parametric, instance-based classifier that assigns labels based on the majority (or weighted) vote of the k nearest training instances in feature space. We used `weights='distance'` to weight neighbors by inverse distance and evaluated a wide range of `k` (2–30) neighbours. Standardization of numerical features was essential because distances define neighborhood relations. A larger `k` (e.g., k=29) produced strong precision by smoothing local noise, at the cost of some recall. Strengths: simple, captures local non-linear structure without training a parametric model. Limitations: storage and computational cost at prediction time, sensitivity to irrelevant features, and potential degradation with synthetic oversampling (SMOTE) that alters neighborhood geometry.

**Hyperparameter meanings**

- **KNN**: `n_neighbors` (number of neighbors used for voting), `weights` (`uniform` or `distance` weighting).

#### 3.4.1 KNN

In [17]:
# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
knn_model = KNeighborsClassifier(weights='distance')
grid_search=GridSearchCV(knn_model, scoring='f1', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_train, y_train)
best_knn_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = best_knn_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors})'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6864    7
1            738  930

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      6871
           1       0.99      0.56      0.71      1668

    accuracy                           0.91      8539
   macro avg       0.95      0.78      0.83      8539
weighted avg       0.92      0.91      0.90      8539



In [18]:
print('N Neighbors', n_neighbors)

N Neighbors 29


#### 3.4.2 KNN w/ Stepwise

In [19]:
# defining the stepwise model
sfs_knn = SequentialFeatureSelector(
    knn_model,
    scoring='f1',
    cv=5
)

# applying the stepwise model
selected_features = sfs_knn.fit(x_train, y_train)

# selecting the stepwised features
x_train_stepwised = x_train[selected_features.get_feature_names_out()]
x_test_stepwised = x_test[selected_features.get_feature_names_out()]

# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
grid_search=GridSearchCV(knn_model, scoring='f1', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_train_stepwised, y_train)
best_knn_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = best_knn_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors}) w/ Stepwise'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6849   22
1            741  927

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      6871
           1       0.98      0.56      0.71      1668

    accuracy                           0.91      8539
   macro avg       0.94      0.78      0.83      8539
weighted avg       0.92      0.91      0.90      8539



In [20]:
print('N Neighbors', n_neighbors)

N Neighbors 28


#### 3.4.3 KNN w/ Stepwise and SMOTE

In [21]:
# stepwising after SMOTE
selected_features_SMOTE = sfs_knn.fit(x_train_SMOTE, y_smote) #select features after SMOTE
x_stepwised_SMOTE = x_train_SMOTE[selected_features_SMOTE.get_feature_names_out()]

# selecting the stepwise columns for the test sample
x_test_stepwised = x_test[x_stepwised_SMOTE.columns]

# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
grid_search=GridSearchCV(knn_model, scoring='f1', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_stepwised_SMOTE, y_smote)
best_knn_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = best_knn_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors}) w/ Stepwise and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5152  1719
1            568  1100

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      6871
           1       0.39      0.66      0.49      1668

    accuracy                           0.73      8539
   macro avg       0.65      0.70      0.65      8539
weighted avg       0.80      0.73      0.75      8539



In [22]:
print('N Neighbors', n_neighbors)

N Neighbors 2


#### 3.5 Decision Tree

Decision Trees partition the feature space with axis-aligned splits to create a tree of decision rules. They natively handle mixed data types and require minimal preprocessing, but are prone to overfitting without constraints (`max_depth`, `min_samples_leaf`, `min_samples_split`). In this study, we tuned split-related hyperparameters to control complexity. Strengths: interpretability and fast predictions; limitations: high variance and unstable splits that benefit from ensemble methods.

**Hyperparameter meanings**
- **Decision Tree**: `max_depth` (maximum depth of the tree), `min_samples_split` (min samples required to split a node), `min_samples_leaf` (min samples required at a leaf node).

#### 3.5.1 Decision Tree

In [23]:
# loading the decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# training the decision tree model
dtree_model.fit(x_train, y_train)

# predicting
y_pred = dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree'] = classification_report(y_test, y_pred, output_dict=True)

Confusion Matrix:
col_0          0     1
fraud_flag            
0           5742  1129
1            615  1053

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      6871
           1       0.48      0.63      0.55      1668

    accuracy                           0.80      8539
   macro avg       0.69      0.73      0.71      8539
weighted avg       0.82      0.80      0.81      8539



#### 3.5.2 Decision Tree w/ Grid Search

In [24]:
# setting the initial hyperparameters
param_grid = {
    'max_depth': [40, 50, 60],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 4]
}

# loading the grid search decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=dtree_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    error_score='raise'
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_dtree_model = grid_search.best_estimator_

# prediction
y_pred = best_dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree w/ GridSearch'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5742  1129
1            615  1053

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      6871
           1       0.48      0.63      0.55      1668

    accuracy                           0.80      8539
   macro avg       0.69      0.73      0.71      8539
weighted avg       0.82      0.80      0.81      8539



In [25]:
print(
    'max_depth', best_dtree_model.max_depth, '\n',
    'min_samples_splt', best_dtree_model.min_samples_split, '\n',
    'min_samples_leaf', best_dtree_model.min_samples_leaf)

max_depth 50 
 min_samples_splt 2 
 min_samples_leaf 1


#### 3.5.3 Decision Tree w/ Grid Search and SMOTE

In [26]:
# setting the initial hyperparameters
param_grid = {
    'max_depth': [35, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# loading the grid search decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=dtree_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train_SMOTE, y_smote)
best_dtree_model = grid_search.best_estimator_

# prediction
y_pred = best_dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree w/ GridSearch and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5534  1337
1            577  1091

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.81      0.85      6871
           1       0.45      0.65      0.53      1668

    accuracy                           0.78      8539
   macro avg       0.68      0.73      0.69      8539
weighted avg       0.82      0.78      0.79      8539



In [27]:
print(
    'max_depth', best_dtree_model.max_depth, '\n',
    'min_samples_splt', best_dtree_model.min_samples_split, '\n',
    'min_samples_leaf', best_dtree_model.min_samples_leaf)

max_depth 50 
 min_samples_splt 2 
 min_samples_leaf 1


#### 3.6 Random Forest

Random Forest aggregates an ensemble of decorrelated decision trees via bootstrap aggregation (bagging) and random feature selection at splits. This reduces variance and improves generalization over single trees. We tuned `n_estimators`, `max_depth`, and sampling parameters; feature importances from the ensemble provided insight into variable relevance. Strengths: robust off-the-shelf performance and resistance to overfitting; limitations: reduced interpretability compared with single trees and sensitivity to class imbalance unless addressed (we used `class_weight` and SMOTE experiments).

**Hyperparameter meanings**

- **Random Forest**: `n_estimators` (number of trees), `max_depth`, `min_samples_split`, `min_samples_leaf`, `bootstrap` (whether to sample with replacement).


#### 3.6.1 Random Forest w/ GridSearch

In [28]:
# setting the initial hyperparameters
param_grid = {
    'n_estimators': [70, 75],
    'max_depth': [35, 45],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# setting the random forest model
rand_for = RandomForestClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=rand_for,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_rand_for_model = grid_search.best_estimator_

# prediction
y_pred = best_rand_for_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Random Forest w/ GridSearch'] = classification_report(y_test, y_pred, output_dict=True)

# selecing the best model out of all that have been run
best_model = best_rand_for_model

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6871    0
1            741  927

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      6871
           1       1.00      0.56      0.71      1668

    accuracy                           0.91      8539
   macro avg       0.95      0.78      0.83      8539
weighted avg       0.92      0.91      0.90      8539



In [29]:
print(
    'n_estimators', best_rand_for_model.n_estimators, '\n'
    'max_depth', best_rand_for_model.max_depth, '\n',
    'min_samples_splt', best_rand_for_model.min_samples_split, '\n',
    'min_samples_leaf', best_rand_for_model.min_samples_leaf, '\n',
    'bootstrap', best_rand_for_model.bootstrap
    )

n_estimators 70 
max_depth 35 
 min_samples_splt 2 
 min_samples_leaf 1 
 bootstrap False


#### 3.6.2 Random Forest w/ GridSearch and SMOTE

In [30]:
# setting the initial hyperparameters
param_grid = {
    'n_estimators': [110, 130],
    'max_depth': [45, 55],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# setting the random forest model
rand_for = RandomForestClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=rand_for,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train_SMOTE, y_smote)
best_rand_for_model = grid_search.best_estimator_

# prediction
y_pred = best_rand_for_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Random Forest w/ GridSearch and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6731  140
1            723  945

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      6871
           1       0.87      0.57      0.69      1668

    accuracy                           0.90      8539
   macro avg       0.89      0.77      0.81      8539
weighted avg       0.90      0.90      0.89      8539



In [31]:
print(
    'n_estimators', best_rand_for_model.n_estimators, '\n'
    'max_depth', best_rand_for_model.max_depth, '\n',
    'min_samples_splt', best_rand_for_model.min_samples_split, '\n',
    'min_samples_leaf', best_rand_for_model.min_samples_leaf, '\n',
    'bootstrap', best_rand_for_model.bootstrap
    )

n_estimators 130 
max_depth 45 
 min_samples_splt 2 
 min_samples_leaf 1 
 bootstrap False


#### 3.7 Effects of SMOTE and Feature Selection

SMOTE synthesizes minority-class observations to balance training data and help classifiers learn decision boundaries for rare fraud cases. However, SMOTE can distort the original feature space—particularly for distance-based methods such as KNN—so we evaluated models with and without SMOTE and reported both results. Sequential feature selection (SFS) reduced dimensionality and removed noisy features, which helped stabilize models that are sensitive to irrelevant variables.

---
## 4. Evaluation Strategy

The success of a fraud detection model cannot be measured by simple accuracy due to the heavily imbalanced nature of the data. Therefore, the evaluation relied on a comprehensive suite of metrics derived from the Confusion Matrix.

### Primary Metric: F1-score (Class 1 - Fraud)

The **F1-score** is the harmonic mean of Precision and Recall. It is the gold standard metric for evaluation in problems with high class imbalance, as it penalizes models that perform poorly in either dimension.

$$\text{F1-score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

* **Rationale:** Our goal is to **balance** the desire to correctly identify fraud cases (high Recall) against the cost of flagging legitimate applications as fraud (high Precision). Maximizing the F1-score ensures that neither Recall nor Precision is ignored.

### Core Metrics (Focus on Fraud — Class 1)

These metrics evaluate the model's performance specifically regarding the detection of the minority class (Fraud):

| Metric | Formula | Explanation | Implication in Fraud Detection |
| :--- | :--- | :--- | :--- |
| **Precision** | $\frac{TP}{TP + FP}$ | The ratio of correctly predicted positive observations to the total predicted positive observations (True Positives + False Positives). | **Quality of Detection:** Out of all applications flagged as fraud, how many were actually fraudulent? High precision reduces false alarms for investigators. |
| **Recall (Sensitivity)** | $\frac{TP}{TP + FN}$ | The ratio of correctly predicted positive observations to all observations in the actual class (True Positives + False Negatives). | **Coverage of Detection:** Out of all actual fraud cases, how many did the model correctly identify? High recall is essential to minimize financial losses. |



### Aggregate Metrics

These metrics provide a summary of overall performance, often including performance across both classes:

| Metric | Explanation | Relevance |
| :--- | :--- | :--- |
| **Accuracy** | The ratio of correctly predicted observations to the total observations. | **Limited Use:** In imbalanced data, a model predicting "No Fraud" 99% of the time can achieve 99% accuracy, making it misleading. It is reported for completeness but not prioritized. |
| **Macro Average** | The unweighted average of the metric (e.g., F1) calculated independently for each class. | **Balanced View:** Gives equal weight to both the majority (Class 0) and minority (Class 1) classes. This is useful for judging a model's performance on the difficult minority class without distortion from the majority class size. |
| **Weighted Average** | The average of the metric (e.g., F1) for each class, weighted by the number of true instances for each class. | **Overall Performance:** Reflects the model's performance relative to the data distribution. Since our data is highly imbalanced, this average will strongly reflect the performance on the majority class. |



---

## 5. Results

Random Forest w/ GridSearch produced the best F1 for fraud (71.40%) and is preferred for production due to ensemble stability and interpretability; KNN (k=29) also achieved very high precision (~99%) but is better considered a secondary, high-precision detector rather than the primary production model.

### 5.1 Error Analysis
- **False Positives**: Random Forest (GridSearch) and KNN both show very low false positive rates; RF reports precision ≈ 1.00 and KNN ≈ 0.99. 
- **False Negatives**: Both RF and KNN have moderate recall (~0.56), meaning they miss roughly 40–45% of fraud cases at the default threshold.

In [32]:
# all models
models = list(results.keys())

all_rows = []

for model in models:
    rep = results[model]  # classification_report dict

    row = {}
    row['model'] = model

    # classes 0 and 1
    for label in ['0', '1']:
        for metric in ['precision', 'recall', 'f1-score']:
            row[f'{label}_{metric}'] = rep[label][metric]

    # accuracy
    row['accuracy'] = rep['accuracy']

    # macro avg
    for metric in ['precision', 'recall', 'f1-score']:
        row[f'macro_{metric}'] = rep['macro avg'][metric]

    # weighted avg
    for metric in ['precision', 'recall', 'f1-score']:
        row[f'weighted_{metric}'] = rep['weighted avg'][metric]

    all_rows.append(row)

df_results = pd.DataFrame(all_rows)

df_results


,model,0_precision,0_recall,0_f1-score,1_precision,1_recall,1_f1-score,accuracy,macro_precision,macro_recall,macro_f1-score,weighted_precision,weighted_recall,weighted_f1-score
0,Logistic Regression,0.801740,0.496143,0.612964,0.192442,0.494604,0.277078,0.495843,0.497091,0.495374,0.445021,0.682721,0.495843,0.547352
1,Logistic Regression w/ Stepwise,0.802617,0.312473,0.449822,0.194407,0.683453,0.302708,0.384940,0.498512,0.497963,0.376265,0.683810,0.384940,0.421085
2,Logistic Regression w/ Stepwise and SMOTE,0.801338,0.278853,0.413734,0.194047,0.715228,0.305271,0.364094,0.497693,0.497040,0.359502,0.682711,0.364094,0.392547
3,KNN (29),0.902920,0.998981,0.948525,0.992529,0.557554,0.714012,0.912753,0.947725,0.778268,0.831268,0.920424,0.912753,0.902715
4,KNN (28) w/ Stepwise,0.902372,0.996798,0.947237,0.976818,0.555755,0.708445,0.910645,0.939595,0.776277,0.827841,0.916914,0.910645,0.900592
5,KNN (2) w/ Stepwise and SMOTE,0.900699,0.749818,0.818362,0.390209,0.659472,0.490305,0.732170,0.645454,0.704645,0.654334,0.800981,0.732170,0.754280
6,Decision Tree,0.903256,0.835686,0.868158,0.482585,0.631295,0.547013,0.795761,0.692921,0.733491,0.707586,0.821083,0.795761,0.805426
7,Decision Tree w/ GridSearch,0.903256,0.835686,0.868158,0.482585,0.631295,0.547013,0.795761,0.692921,0.733491,0.707586,0.821083,0.795761,0.805426
8,Decision Tree w/ GridSearch and SMOTE,0.905580,0.805414,0.852565,0.449341,0.654077,0.532715,0.775852,0.677461,0.729745,0.692640,0.816459,0.775852,0.790086
9,Random Forest w/ GridSearch,0.902654,1.000000,0.948837,1.000000,0.555755,0.714451,0.913222,0.951327,0.777878,0.831644,0.921669,0.913222,0.903052


## 6. Conclusions
- Random Forest (GridSearch) achieved the best F1 for fraud detection and is recommended as the primary model for deployment; KNN (k=29) is a high-precision alternative and useful in ensemble strategies.

The perspective of choose the best model by the F1-Score it's relevant and reasonable and returns the best intermediary result considering both Precision and Recall. 

Even though, on the business perspective of fraudulent bank loan applications, a high precision on class "1" (flag a true fraud as such) may be less important than to have a high recall on class "1" (identify more frauds, even though with mroe "false alarms").

In a scenario of bank fraud detection, it's common that the classifier algorithm be a first security layer raising a concern so then it be reviewed with more details by a human being. Therefore, a higher occurence of false positives (lower precision) would increase the customer's approval time and costs with human analysis, but if in exchange for a higher number of true positives detected (higher recall), it would reduce the losses with fraudulent applications.

In resume, for this specific situation, score the models by Recall instead of F1-Score may result in a better financial return on reducing the losses on fraudulent applications.

# Test

## Data Treatment

### Data Modelling

In [33]:
df_test = pd.read_csv('test.csv')
df_test = df_test.set_index('Unnamed: 0')
df_test = df_test.drop(columns=['data_batch_id']) ##removing first column, that looks just an random id

# df_test = df_test.drop_duplicates() # there's duplicates but if removed, kaggle doesn't accept the submission

# removing unwanted object features
categ_cols = ['purpose_of_loan', 'employment_status', 'property_ownership_status']
df_test[categ_cols] = df_test[categ_cols].astype('category')

# capitalizing category features
for column in categ_cols:
    df_test[column] = df_test[column].str.capitalize()
    df_test[column] = df_test[column].str.strip()

# removing untrustable dummy columns
dummyDrop = [col for col in df_test.columns if 'loan_type' in col]
df_test = df_test.drop(columns=dummyDrop)

### Outliers Removal

This step step was not applied since the Kaggle submission demands 11k rows

In [34]:
'''# removing outliers by the loan tenure months
q1 = df_test['loan_tenure_months'].quantile(0.25)
q3 = df_test['loan_tenure_months'].quantile(0.75)
iqr = q3-q1
upper_bound = q3 + iqr*1.5

# removing outliers
df_test = df_test[df_test['loan_tenure_months']>=upper_bound]'''

"# removing outliers by the loan tenure months\nq1 = df_test['loan_tenure_months'].quantile(0.25)\nq3 = df_test['loan_tenure_months'].quantile(0.75)\niqr = q3-q1\nupper_bound = q3 + iqr*1.5\n\n# removing outliers\ndf_test = df_test[df_test['loan_tenure_months']>=upper_bound]"

### Handling Missing Values

#### Gender: Imputation

In [35]:
## gender imputation through KNN
def gender_code(row):
    if row['gender_Male'] == 1:
        return 1
    elif row['gender_Other'] == 1:
        return 0
    else:
        None #columns as neither male or other will be treated as unkown

df_test['gender_code'] = df_test.apply(gender_code, axis=1)

# defining known and unknown gender
df_known = df_test[df_test['gender_code'].notna()]
df_unknown = df_test[df_test['gender_code'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['gender_code']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_test.loc[df_test['gender_code'].isna(), 'gender_code'] = knn.predict(x_test_scaled)

# recreating the gender label, where "0" is man and "1" is woman
gender_drop = ['gender_Male', 'gender_Other']
df_test = df_test.drop(columns=gender_drop)
df_test = df_test.rename(columns={'gender_code': 'gender_male'})

# 0: woman ; 1: man
df_test['gender_male'].value_counts()

gender_male
0.0    5547
1.0    5453
Name: count, dtype: int64

#### Number of Dependents: Imputation

In [36]:
# defining known and unknown number of dependents
df_known = df_test[df_test['number_of_dependents'].notna()]
df_unknown = df_test[df_test['number_of_dependents'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['number_of_dependents']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_test.loc[df_test['number_of_dependents'].isna(), 'number_of_dependents'] = knn.predict(x_test_scaled)

### Dropping

In [37]:
df_test = df_test.drop(columns=['monthly_income', # missing values and redundant with 'yearly_income'
                                  'cibil_score', # perfectly normal, likely not real
                                  'loan_amount_requested', # redundant with 'loan_amount_usd'
                                  'application_date', # decided to not be used
                                  'application_id', # irrelevant for prediction
                                  'customer_id', # irrelevant for prediction
                                  'residential_address', # decided to not be used
                                  ])


# removing untrustable dummy columns
dummyDrop = [col for col in df_test.columns if 'loan_type' in col]
df_train = df_test.drop(columns=dummyDrop)

### Dummying

In [38]:
# transforming categorical features into dummy ones, removing one of the categories to avoid colinearility
df_test = pd.get_dummies(df_test, columns=categ_cols, drop_first=True, dtype=int)

### Scaling

In [39]:
# setting dummy columns
dummy_cols = [col for col in df_test if any(sub in col for sub in categ_cols) or col == 'gender_male']
not_dummy_cols = [col for col in df_test if col not in dummy_cols] 

# scaling feature columns but dummy ones
x_scaled = pd.DataFrame(
    scaler_train.transform(df_test[not_dummy_cols]), # transform based on the train model
    columns=not_dummy_cols, 
    index=df_test.index)

# concatenating the dummy cols in the scaled numeric features
x_final = pd.concat([x_scaled, df_test[dummy_cols]], axis=1)

### Prediction on Best Model

In [40]:
# the smoted data should be used only for training, not for tests
df_test = df_test.copy()
df_test['fraud_flag'] = best_model.predict(x_final)

### Exporting Prediction for Kaggle

In [41]:
df_test.index.name = "ID"
df_test['fraud_flag'].to_csv('submission_f1.csv', index=True)